# **Practical Session 3: Pacman with AI**

### Introducción

##### En esta práctica vamos a implementar un agente inteligente capaz de jugar y ganar al juego de Pacman. Este agente tendrá una red neuronal entrenada con partidas reales jugadas por nosotros y también constará de diferentes heurísticas y algoritmos (AlphaBeta) que mejorará el rendimiento.
##### El objetivo de esta práctica es implementar correctamente el algoritmo AlphaBeta con la red neuronal y nuestras heurísticas. De esta manera conseguiríamos que nuestro agente tome buenas decisiones durante el desarrollo de la partida y acabe ganando.

### Part 1 — New heuristics:

#### Antes de ver las nuevas heurísticas, cabe destacar que hemos modificado ligeramente las ya implementadas. 
##### Sobre la primera heurística, se ha aumentado ligeramente la puntuación que aporta: de *score += 1.0 / (min_food_distance + 1)* a *score += 2.0 / (min_food_distance + 1)*

##### En cuanto a la segunda, se han modificado los valores para castigar más y más fácilmente el hecho de estar cerca de un fantasma: de *if ghost_distance <= 2: score -= 200* a *if ghost_distance <= 3: score -= 300*

#### **Heurística 1**: Manejo de las cápsulas
##### Va a calcular la distancia mínima de pacman a cualquiera de las cápsulas (Los puntos grandes que le hacen poder comerse a los fantasmas) si hay alguna y va a recompensar al agente por estar cerca de alguna si también hay algún fantasma cerca. Si el fantasma está muy cerca, la recompensa aumenta para potenciar el efecto de esta heurística. Si el fantasma está cerca y está lejos de una cápsula, castiga al agente.

#### **Heurística 2**: Penalización por comida restante
##### Va a penalizar ligeramente al agente por cada punto de comida restante en el mapa, así debería priorizar comerse la comida para perder menos puntuación. Además, va a penalizar en función a la distancia media a la comida, esto debería hacer que pacman se mueva a zonas con mucha comida.

### Part 2 — Network training:

##### Nuestro dataset se compondría de 40 partidas jugadas por nosotros, siendo todas ellas victorias. No hemos seguido exactamente una misma jugada siempre para que no se entrene con un solo camino. Además hemos intentado minimizar el tiempo en el que nos movemos por espacios vacíos sin comida.

### Part 3 — AlphaBetaNeuralAgent:

##### Para nuestra implementación de AlphaBetaNeuralAgent hemos incluido como parámetros modificables w_heuristic y w_neural, que son los pesos de las heurísticas y de la red neuronal respectivamente. Estos parámetros los hemos definido como 0.5 y 0.5 por defecto. Estos valores nos daría un resultado equilibrado entre ambos valores. De esta manera podemos controlar lo que influye cada parte del proceso en la decisión final.

##### La función que utiliza estos valores es evaluation_combined que se encarga de obtener una puntuación total teniendo en cuenta tanto la puntuación obtenida por la red neuronal como la obtenida por las heurísticas.

##### Luego tenemos la función getAction que se encarga de crear las 3 funciones clave para AlphaBeta. Esas funciones son: alphabeta, maxValue y minValue.

##### **alphabeta(agentIndex, depth, state, alpha, beta)**: Esta función se encarga de seguir la lógica básica de AlphaBeta. Si es el turno de Pacman (agente 0) llama a maxValue y si es el turno de uno de los fantasmas (agente >= 1) llama a MinValue. Por último si completa la profundidad definida o termina la partida (ya sea por victoria o por derrota) llama a la funcion evaluation_combined ya explicada anteriormente.

##### **maxValue(agentIndex, depth, state, alpha, beta)**: Esta función va a elegir la opción que tomaría Pacman, es decir, la que proporcionaría una mayor puntuación.

##### **minValue(agentIndex, depth, state, alpha, beta)**: Al contrario que maxValue, esta función elegirá la opción que tomarían los fantasmas en sus turnos, es decir, la que aporte menor puntuación.

##### Por último, utilizamos un bucle que recorre las acciones legales y llama a la función alphabeta. De esta manera, almacenará cuál sería la mejor acción y la puntuación que aportaría. Finalmente, se devolvería esa mejor acción.

### Part 4 — Results:

##### (Por ahora lo de esta tabla es con las 40 partidas mías guardadas, si añadimos más, se vuelve a calcular la tabla)
##### (Hay 2 "Greedy" por que me salieron valores dinstintos, hay más partidas, tiene sentido, dejo la comparación porque me parece raro esa diferencia, para que la veas y veamos si cambiar codigo, volver a calcular,...)

In [2]:
print(
"""
Mapas utilizados:


- Classic Layout:

%%%%%%%%%%%%%%%%%%%%
%o...%........%....%
%.%%.%.%%%%%%.%.%%.%
%.%..............%.%
%.%.%%.%%  %%.%%.%.%
%......%G  G%......%
%.%.%%.%%%%%%.%%.%.%
%.%..............%.%
%.%%.%.%%%%%%.%.%%.%
%....%...P....%...o%
%%%%%%%%%%%%%%%%%%%%


- New Layout

%%%%%%%%%%%%%%%%%%%%
%o... ........ ....%
%.%%. .%%..%%. .%%.%
%.%..............%.%
%.%.%%.%%  %%.%%.%.%
%......%G  G%......%
%.%.%%.%%%%%%.%%.%.%
%.%..............%.%
%.%%.%.%%..%%.%.%%.%
%....%...P....%...o%
%%%%%%%%%%%%%%%%%%%%
"""
)


Mapas utilizados:


- Classic Layout:

%%%%%%%%%%%%%%%%%%%%
%o...%........%....%
%.%%.%.%%%%%%.%.%%.%
%.%..............%.%
%.%.%%.%%  %%.%%.%.%
%......%G  G%......%
%.%.%%.%%%%%%.%%.%.%
%.%..............%.%
%.%%.%.%%%%%%.%.%%.%
%....%...P....%...o%
%%%%%%%%%%%%%%%%%%%%


- New Layout

%%%%%%%%%%%%%%%%%%%%
%o... ........ ....%
%.%%. .%%..%%. .%%.%
%.%..............%.%
%.%.%%.%%  %%.%%.%.%
%......%G  G%......%
%.%.%%.%%%%%%.%%.%.%
%.%..............%.%
%.%%.%.%%..%%.%.%%.%
%....%...P....%...o%
%%%%%%%%%%%%%%%%%%%%



| Configuration                           | Classic Layout | New layout |
| --------------------------------------- | -------------- | ---------- |
| Greedy neural agent (no modifications)  |    54.3 / 0%   | 190.1 / 0% |
| Greedy neural agent (no modifications)  |	  -90.8 / 0%   |   2.8 / 0% |
| Greedy neural + nuevas heurísticas      |  -54.5  / 0%   |-137.9 / 0% |
| Alpha-Beta + Neural + Heurísticas       |   541.1 / 20%  | 599.1 / 20%|

### Analisis

Primero, se nota la mejora con la última configuración, Alpha-Beta + Neutral + Heurística. Mientras que los agentes Greedy sin mejoras nunca logran ganar, el algoritmo Alpha-Beta con profundidad 2 consigue victorias tanto en el mapa clásico como en el nuevo. Esto demuestra que "predecir el futuro" permite al agente evitar emboscadas, riesgos o peligros que la red neuronal por sí sola no detecta.

Dentro del mapa clásico, se observa una mejora del agente con la implementación de las heurísticas, de -90.8 a -54.5. Esto confirma que las nuevas heurísticas, ayudan a situar mejor los estados, aunque sigan sin ser suficientes para ganar sin una búsqueda a futuro. Una cosa muy interasnte que vemos es cómo la Heurística empeora al agente, en "New Layout", esto se puede deber a cómo está construido este mapa, pudiendo hacer que las funciones entren en conflicto, ademas de que este mapa tiene menos partidas guardadas para su entrenamiento, siendo un "territorio completamente desconocido", dándonos peores resultados.

Un dato muy importante es que el agente Alpha-Beta ha funcionado incluso mejor en puntuación media en el nuevo mapa que en el clásico, lo que indica que el sistema es capaz de adaptarse y funcionar bien a entornos que no ha visto durante el entrenamiento.

### Conclusions